In [ ]:
import pandas as pd
import numpy as np
import optuna
import shap
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

def prepare_features(train_df, test_df, feature_cols, target_col):
    X_train = train_df[feature_cols].copy()
    y_train = train_df[target_col].copy()
    X_test = test_df[feature_cols].copy()
    y_test = test_df[target_col].copy()

    imputer = SimpleImputer(strategy='median')
    X_train_imp = imputer.fit_transform(X_train)
    X_test_imp = imputer.transform(X_test)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_imp)
    X_test_scaled = scaler.transform(X_test_imp)

    return X_train_scaled, X_test_scaled, y_train, y_test, imputer, scaler

def evaluate_model(y_true, y_pred, name=""):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    underestimations = (y_pred < y_true).sum()
    svr = underestimations / len(y_true) * 100
    overestimations = (y_pred > y_true).sum()
    ovr = overestimations / len(y_true) * 100
    print(f"\n{'='*50}")
    print(f"{name}")
    print(f"{'='*50}")
    print(f"MAE:  {mae:.6f}")
    print(f"RMSE: {rmse:.6f}")
    print(f"R2:   {r2:.6f}")
    print(f"SVR (недооценка): {svr:.2f}% ({underestimations}/{len(y_true)})")
    print(f"OVR (переоценка): {ovr:.2f}% ({overestimations}/{len(y_true)})")
    return {'MAE': mae, 'RMSE': rmse, 'R2': r2, 'SVR': svr, 'OVR': ovr}

def explain_with_shap(model, X_sample, feature_names):
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_sample)
    plt.figure(figsize=(12, 5))
    shap.summary_plot(shap_values, X_sample, feature_names=feature_names, show=False)
    plt.tight_layout()
    plt.show()
    importance_df = pd.DataFrame({'feature': feature_names, 'shap_importance': np.abs(shap_values).mean(axis=0)}).sort_values('shap_importance', ascending=False)
    return importance_df, shap_values

def objective_random_forest(trial, X_train, y_train, X_val, y_val):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500, step=50),
        'max_depth': trial.suggest_int('max_depth', 5, 20),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None]),
        'random_state': 42,
        'n_jobs': -1
    }
    model = RandomForestRegressor(**params)
    model.fit(X_train, y_train)
    pred = model.predict(X_val)
    return mean_absolute_error(y_val, pred)

def tune_random_forest(X_train, y_train, X_val, y_val, n_trials=20):
    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(lambda trial: objective_random_forest(trial, X_train, y_train, X_val, y_val), n_trials=n_trials, show_progress_bar=True)
    return study.best_params, study.best_value

def objective_catboost(trial, X_train, y_train, X_val, y_val):
    params = {
        'iterations': trial.suggest_int('iterations', 200, 600, step=50),
        'depth': trial.suggest_int('depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
        'random_seed': 42,
        'verbose': False
    }
    model = CatBoostRegressor(**params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], early_stopping_rounds=50, verbose=False)
    pred = model.predict(X_val)
    return mean_absolute_error(y_val, pred)

def tune_catboost(X_train, y_train, X_val, y_val, n_trials=15):
    study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(lambda trial: objective_catboost(trial, X_train, y_train, X_val, y_val), n_trials=n_trials, show_progress_bar=True)
    return study.best_params, study.best_value

def run_linear_regression(X_train, X_test, y_train, y_test, name):
    lr = LinearRegression()
    lr.fit(X_train, y_train)
    pred = lr.predict(X_test)
    evaluate_model(y_test, pred, name)
    return pred

def run_ridge_regression(X_train, X_test, y_train, y_test, alpha, name):
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train, y_train)
    pred = ridge.predict(X_test)
    evaluate_model(y_test, pred, name)
    return pred

def run_random_forest_default(X_train, X_test, y_train, y_test, name):
    rf = RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    pred = rf.predict(X_test)
    evaluate_model(y_test, pred, name)
    return pred

def run_catboost_default(X_train, X_test, y_train, y_test, name):
    cb = CatBoostRegressor(iterations=300, depth=6, learning_rate=0.05, verbose=False, random_seed=42)
    cb.fit(X_train, y_train)
    pred = cb.predict(X_test)
    evaluate_model(y_test, pred, name)
    return pred

def run_tuned_random_forest(X_train, X_test, y_train, y_test, X_tr, y_tr, X_val, y_val, name, n_trials=15):
    best_params, best_val = tune_random_forest(X_tr, y_tr, X_val, y_val, n_trials)
    print(f"Best params: {best_params}")
    print(f"Best val MAE: {best_val:.6f}")
    rf = RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    pred = rf.predict(X_test)
    evaluate_model(y_test, pred, name)
    return pred, best_params

def run_tuned_catboost(X_train, X_test, y_train, y_test, X_tr, y_tr, X_val, y_val, name, n_trials=15):
    best_params, best_val = tune_catboost(X_tr, y_tr, X_val, y_val, n_trials)
    print(f"Best params: {best_params}")
    print(f"Best val MAE: {best_val:.6f}")
    cb = CatBoostRegressor(**best_params, verbose=False, random_seed=42)
    cb.fit(X_train, y_train)
    pred = cb.predict(X_test)
    evaluate_model(y_test, pred, name)
    return pred, best_params

def run_ensemble(pred1, pred2, y_test, name):
    pred_ensemble = (pred1 + pred2) / 2
    evaluate_model(y_test, pred_ensemble, name)
    return pred_ensemble

def run_ensemble_three(pred1, pred2, pred3, y_test, name):
    pred_ensemble = (pred1 + pred2 + pred3) / 3
    evaluate_model(y_test, pred_ensemble, name)
    return pred_ensemble

In [ ]:
train_df = pd.read_parquet('FE_google/features_result/train_df_google.parquet')
test_df = pd.read_parquet('FE_google/features_result/test_df_google.parquet')

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")


feature_cols = [
    'maximum_cpu', 'maximum_memory',
    'avg_cpu_mean_0_1d', 'avg_cpu_mean_1_2d', 'avg_cpu_mean_2_3d', 'avg_cpu_mean_3_4d',
    'avg_cpu_lag_5m', 'avg_cpu_lag_10m', 'avg_cpu_lag_15m', 'avg_cpu_lag_20m',
    'avg_cpu_autocorr_lag1_before', 'avg_cpu_history_mean_before', 'avg_cpu_history_std_before',
    'requested_cpu', 'requested_ram'
]

print(f"Используется признаков: {len(feature_cols)}")
print(f"Признаки: {feature_cols}")

In [ ]:
=== ЗАДАЧА 1: ПРОГНОЗ НА 5 МИНУТ (target_avg_cpu_5m) ===

Train size: 797581, Test size: 199695

Linear Regression (базовая модель):

==================================================
Linear Regression (5min)
==================================================
MAE:  0.084847
RMSE: 0.130741
R2:   0.802836
SVR (недооценка): 48.84% (97541/199695)
OVR (переоценка): 51.16% (102154/199695)

Random Forest (базовые параметры):

==================================================
Random Forest (5min) - default
==================================================
MAE:  0.050601
RMSE: 0.098325
R2:   0.888486
SVR (недооценка): 45.80% (91460/199695)
OVR (переоценка): 54.20% (108235/199695)

CatBoost (базовые параметры):
[I 2026-05-28 21:28:52,286] A new study created in memory with name: no-name-eda4a55d-c893-4a3d-a310-713fa4ce3fd1

==================================================
CatBoost (5min) - default
==================================================
MAE:  0.054745
RMSE: 0.101541
R2:   0.881071
SVR (недооценка): 48.52% (96893/199695)
OVR (переоценка): 51.48% (102802/199695)

=== Optuna тюнинг для Random Forest ===
Best trial: 0. Best value: 0.0516737:   7%|▋         | 1/15 [01:05<15:20, 65.75s/it]
[I 2026-05-28 21:29:58,034] Trial 0 finished with value: 0.05167365379115855 and parameters: {'n_estimators': 250, 'max_depth': 20, 'min_samples_split': 15, 'min_samples_leaf': 6, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.05167365379115855.
Best trial: 0. Best value: 0.0516737:  13%|█▎        | 2/15 [02:44<18:24, 84.94s/it]
[I 2026-05-28 21:31:36,413] Trial 1 finished with value: 0.05349092422690875 and parameters: {'n_estimators': 450, 'max_depth': 14, 'min_samples_split': 15, 'min_samples_leaf': 1, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.05167365379115855.
Best trial: 0. Best value: 0.0516737:  20%|██        | 3/15 [04:21<18:07, 90.66s/it]
[I 2026-05-28 21:33:13,875] Trial 2 finished with value: 0.060612264658774055 and parameters: {'n_estimators': 150, 'max_depth': 7, 'min_samples_split': 7, 'min_samples_leaf': 6, 'max_features': None}. Best is trial 0 with value: 0.05167365379115855.
Best trial: 0. Best value: 0.0516737:  27%|██▋       | 4/15 [04:47<11:54, 64.96s/it]
[I 2026-05-28 21:33:39,438] Trial 3 finished with value: 0.06011909097952529 and parameters: {'n_estimators': 150, 'max_depth': 9, 'min_samples_split': 8, 'min_samples_leaf': 5, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.05167365379115855.
Best trial: 0. Best value: 0.0516737:  33%|███▎      | 5/15 [07:41<17:24, 104.45s/it]
[I 2026-05-28 21:36:33,897] Trial 4 finished with value: 0.07006218330998952 and parameters: {'n_estimators': 350, 'max_depth': 5, 'min_samples_split': 13, 'min_samples_leaf': 2, 'max_features': None}. Best is trial 0 with value: 0.05167365379115855.
Best trial: 0. Best value: 0.0516737:  40%|████      | 6/15 [13:20<27:37, 184.11s/it]
[I 2026-05-28 21:42:12,660] Trial 5 finished with value: 0.05621933457836618 and parameters: {'n_estimators': 450, 'max_depth': 9, 'min_samples_split': 3, 'min_samples_leaf': 7, 'max_features': None}. Best is trial 0 with value: 0.05167365379115855.
Best trial: 6. Best value: 0.0505074:  47%|████▋     | 7/15 [15:20<21:45, 163.17s/it]
[I 2026-05-28 21:44:12,720] Trial 6 finished with value: 0.05050743071730874 and parameters: {'n_estimators': 100, 'max_depth': 19, 'min_samples_split': 6, 'min_samples_leaf': 7, 'max_features': None}. Best is trial 6 with value: 0.05050743071730874.
Best trial: 6. Best value: 0.0505074:  53%|█████▎    | 8/15 [18:16<19:31, 167.32s/it]
[I 2026-05-28 21:47:08,933] Trial 7 finished with value: 0.05052064217016848 and parameters: {'n_estimators': 150, 'max_depth': 20, 'min_samples_split': 16, 'min_samples_leaf': 10, 'max_features': None}. Best is trial 6 with value: 0.05050743071730874.
Best trial: 6. Best value: 0.0505074:  60%|██████    | 9/15 [19:28<13:44, 137.47s/it]
[I 2026-05-28 21:48:20,769] Trial 8 finished with value: 0.058046769150851045 and parameters: {'n_estimators': 100, 'max_depth': 8, 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_features': None}. Best is trial 6 with value: 0.05050743071730874.
Best trial: 6. Best value: 0.0505074:  67%|██████▋   | 10/15 [22:39<12:49, 153.86s/it]
[I 2026-05-28 21:51:31,318] Trial 9 finished with value: 0.05623099798790026 and parameters: {'n_estimators': 250, 'max_depth': 9, 'min_samples_split': 12, 'min_samples_leaf': 2, 'max_features': None}. Best is trial 6 with value: 0.05050743071730874.
Best trial: 6. Best value: 0.0505074:  73%|███████▎  | 11/15 [23:49<08:33, 128.33s/it]
[I 2026-05-28 21:52:41,760] Trial 10 finished with value: 0.052750885883859434 and parameters: {'n_estimators': 300, 'max_depth': 16, 'min_samples_split': 20, 'min_samples_leaf': 9, 'max_features': 'log2'}. Best is trial 6 with value: 0.05050743071730874.
Best trial: 6. Best value: 0.0505074:  80%|████████  | 12/15 [24:16<04:52, 97.41s/it]
[I 2026-05-28 21:53:08,459] Trial 11 finished with value: 0.05213859602265144 and parameters: {'n_estimators': 100, 'max_depth': 20, 'min_samples_split': 20, 'min_samples_leaf': 10, 'max_features': 'log2'}. Best is trial 6 with value: 0.05050743071730874.
Best trial: 6. Best value: 0.0505074:  87%|████████▋ | 13/15 [27:53<04:27, 133.72s/it]
[I 2026-05-28 21:56:45,709] Trial 12 finished with value: 0.05069259464831191 and parameters: {'n_estimators': 200, 'max_depth': 17, 'min_samples_split': 7, 'min_samples_leaf': 8, 'max_features': None}. Best is trial 6 with value: 0.05050743071730874.
Best trial: 6. Best value: 0.0505074:  93%|█████████▎| 14/15 [29:48<02:08, 128.11s/it]
[I 2026-05-28 21:58:40,871] Trial 13 finished with value: 0.050667832736214405 and parameters: {'n_estimators': 100, 'max_depth': 18, 'min_samples_split': 17, 'min_samples_leaf': 10, 'max_features': None}. Best is trial 6 with value: 0.05050743071730874.
Best trial: 6. Best value: 0.0505074: 100%|██████████| 15/15 [32:58<00:00, 131.88s/it]
[I 2026-05-28 22:01:50,544] Trial 14 finished with value: 0.052094616063991496 and parameters: {'n_estimators': 200, 'max_depth': 13, 'min_samples_split': 9, 'min_samples_leaf': 8, 'max_features': None}. Best is trial 6 with value: 0.05050743071730874.
Best params: {'n_estimators': 100, 'max_depth': 19, 'min_samples_split': 6, 'min_samples_leaf': 7, 'max_features': None}
Best val MAE: 0.050507
[I 2026-05-28 22:04:15,591] A new study created in memory with name: no-name-8128f74e-1691-40ad-949f-79223e2a3fd6

==================================================
Random Forest (5min) - tuned
==================================================
MAE:  0.049789
RMSE: 0.097393
R2:   0.890589
SVR (недооценка): 45.81% (91471/199695)
OVR (переоценка): 53.96% (107763/199695)

=== Optuna тюнинг для CatBoost ===
Best trial: 0. Best value: 0.0519761:   7%|▋         | 1/15 [00:16<03:51, 16.53s/it]
[I 2026-05-28 22:04:32,119] Trial 0 finished with value: 0.05197607867947459 and parameters: {'iterations': 350, 'depth': 10, 'learning_rate': 0.05395030966670229, 'l2_leaf_reg': 6.387926357773329}. Best is trial 0 with value: 0.05197607867947459.
Best trial: 0. Best value: 0.0519761:  13%|█▎        | 2/15 [00:21<02:08,  9.92s/it]
[I 2026-05-28 22:04:37,405] Trial 1 finished with value: 0.07134699335523163 and parameters: {'iterations': 250, 'depth': 5, 'learning_rate': 0.011430983876313222, 'l2_leaf_reg': 8.795585311974417}. Best is trial 0 with value: 0.05197607867947459.
Best trial: 0. Best value: 0.0519761:  20%|██        | 3/15 [00:34<02:16, 11.37s/it]
[I 2026-05-28 22:04:50,505] Trial 2 finished with value: 0.057917763227283134 and parameters: {'iterations': 450, 'depth': 8, 'learning_rate': 0.010485387725194618, 'l2_leaf_reg': 9.72918866945795}. Best is trial 0 with value: 0.05197607867947459.
Best trial: 0. Best value: 0.0519761:  27%|██▋       | 4/15 [00:45<02:03, 11.22s/it]
[I 2026-05-28 22:05:01,503] Trial 3 finished with value: 0.05931488274866212 and parameters: {'iterations': 550, 'depth': 5, 'learning_rate': 0.015199348301309814, 'l2_leaf_reg': 2.650640588680904}. Best is trial 0 with value: 0.05197607867947459.
Best trial: 0. Best value: 0.0519761:  33%|███▎      | 5/15 [00:53<01:38,  9.87s/it]
[I 2026-05-28 22:05:08,979] Trial 4 finished with value: 0.05638901276122744 and parameters: {'iterations': 300, 'depth': 7, 'learning_rate': 0.027036160666620016, 'l2_leaf_reg': 3.6210622617823773}. Best is trial 0 with value: 0.05197607867947459.
Best trial: 0. Best value: 0.0519761:  40%|████      | 6/15 [01:01<01:24,  9.36s/it]
[I 2026-05-28 22:05:17,331] Trial 5 finished with value: 0.06135643634187074 and parameters: {'iterations': 450, 'depth': 4, 'learning_rate': 0.019594972058679168, 'l2_leaf_reg': 4.297256589643226}. Best is trial 0 with value: 0.05197607867947459.
Best trial: 0. Best value: 0.0519761:  47%|████▋     | 7/15 [01:16<01:27, 10.98s/it]
[I 2026-05-28 22:05:31,669] Trial 6 finished with value: 0.055331597842035105 and parameters: {'iterations': 400, 'depth': 9, 'learning_rate': 0.015837031559118753, 'l2_leaf_reg': 5.628109945722504}. Best is trial 0 with value: 0.05197607867947459.
Best trial: 0. Best value: 0.0519761:  53%|█████▎    | 8/15 [01:24<01:10, 10.09s/it]
[I 2026-05-28 22:05:39,859] Trial 7 finished with value: 0.0582446810495127 and parameters: {'iterations': 450, 'depth': 4, 'learning_rate': 0.04050837781329675, 'l2_leaf_reg': 2.5347171131856236}. Best is trial 0 with value: 0.05197607867947459.
Best trial: 0. Best value: 0.0519761:  60%|██████    | 9/15 [01:33<00:59,  9.89s/it]
[I 2026-05-28 22:05:49,299] Trial 8 finished with value: 0.052179283966180615 and parameters: {'iterations': 200, 'depth': 10, 'learning_rate': 0.0923915031962725, 'l2_leaf_reg': 8.275576133048151}. Best is trial 0 with value: 0.05197607867947459.
Best trial: 0. Best value: 0.0519761:  67%|██████▋   | 10/15 [01:39<00:42,  8.55s/it]
[I 2026-05-28 22:05:54,847] Trial 9 finished with value: 0.05915560594963705 and parameters: {'iterations': 300, 'depth': 4, 'learning_rate': 0.04833180632488466, 'l2_leaf_reg': 4.961372443656412}. Best is trial 0 with value: 0.05197607867947459.
Best trial: 10. Best value: 0.0508896:  73%|███████▎  | 11/15 [02:06<00:57, 14.32s/it]
[I 2026-05-28 22:06:22,265] Trial 10 finished with value: 0.05088958879158249 and parameters: {'iterations': 600, 'depth': 10, 'learning_rate': 0.07224943768187977, 'l2_leaf_reg': 6.758183738140614}. Best is trial 10 with value: 0.05088958879158249.
Best trial: 11. Best value: 0.0508823:  80%|████████  | 12/15 [02:34<00:55, 18.35s/it]
[I 2026-05-28 22:06:49,839] Trial 11 finished with value: 0.050882302374377096 and parameters: {'iterations': 600, 'depth': 10, 'learning_rate': 0.07515866857795253, 'l2_leaf_reg': 6.741745796899547}. Best is trial 11 with value: 0.050882302374377096.
Best trial: 11. Best value: 0.0508823:  87%|████████▋ | 13/15 [02:50<00:35, 17.82s/it]
[I 2026-05-28 22:07:06,436] Trial 12 finished with value: 0.05154557639920064 and parameters: {'iterations': 600, 'depth': 8, 'learning_rate': 0.09858632185164357, 'l2_leaf_reg': 7.0164298800690945}. Best is trial 11 with value: 0.050882302374377096.
Best trial: 11. Best value: 0.0508823:  93%|█████████▎| 14/15 [03:11<00:18, 18.79s/it]
[I 2026-05-28 22:07:27,455] Trial 13 finished with value: 0.05134576300898166 and parameters: {'iterations': 600, 'depth': 9, 'learning_rate': 0.06507547631899815, 'l2_leaf_reg': 7.227974223127852}. Best is trial 11 with value: 0.050882302374377096.
Best trial: 11. Best value: 0.0508823: 100%|██████████| 15/15 [03:37<00:00, 14.51s/it]
[I 2026-05-28 22:07:53,217] Trial 14 finished with value: 0.050977691414387724 and parameters: {'iterations': 550, 'depth': 10, 'learning_rate': 0.07105089399830998, 'l2_leaf_reg': 7.836931920718491}. Best is trial 11 with value: 0.050882302374377096.
Best params: {'iterations': 600, 'depth': 10, 'learning_rate': 0.07515866857795253, 'l2_leaf_reg': 6.741745796899547}
Best val MAE: 0.050882

==================================================
CatBoost (5min) - tuned
==================================================
MAE:  0.050230
RMSE: 0.096057
R2:   0.893570
SVR (недооценка): 48.06% (95971/199695)
OVR (переоценка): 51.94% (103724/199695)

Ensemble (Random Forest tuned + CatBoost tuned):

==================================================
Ensemble (5min)
==================================================
MAE:  0.049508
RMSE: 0.096014
R2:   0.893666
SVR (недооценка): 47.33% (94508/199695)
OVR (переоценка): 52.67% (105187/199695)

SHAP Analysis (CatBoost tuned):

Top 10 features:
                         feature  shap_importance
0                    maximum_cpu         0.080865
6                 avg_cpu_lag_5m         0.054637
2              avg_cpu_mean_0_1d         0.030096
9                avg_cpu_lag_20m         0.018702
1                 maximum_memory         0.016998
12    avg_cpu_history_std_before         0.014932
8                avg_cpu_lag_15m         0.013033
7                avg_cpu_lag_10m         0.011739
10  avg_cpu_autocorr_lag1_before         0.008560
3              avg_cpu_mean_1_2d         0.007774

In [ ]:
print("=== ЗАДАЧА 2: ПРОГНОЗ МАКСИМУМА ЗА 24 ЧАСА (target_avg_cpu_24h) ===\n")

target_col = 'target_avg_cpu_24h'

X_train, X_test, y_train, y_test, imputer, scaler = prepare_features(train_df, test_df, feature_cols, target_col)

print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")

X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

print("\nLinear Regression (базовая модель):")
pred_lr = run_linear_regression(X_train, X_test, y_train, y_test, "Linear Regression (24h)")

print("\nRandom Forest (базовые параметры):")
pred_rf_default = run_random_forest_default(X_train, X_test, y_train, y_test, "Random Forest (24h) - default")

print("\nCatBoost (базовые параметры):")
pred_cb_default = run_catboost_default(X_train, X_test, y_train, y_test, "CatBoost (24h) - default")

print("\n=== Optuna тюнинг для Random Forest ===")
pred_rf_tuned, best_params_rf = run_tuned_random_forest(X_train, X_test, y_train, y_test, X_tr, y_tr, X_val, y_val, "Random Forest (24h) - tuned", n_trials=15)

print("\n=== Optuna тюнинг для CatBoost ===")
pred_cb_tuned, best_params_cb = run_tuned_catboost(X_train, X_test, y_train, y_test, X_tr, y_tr, X_val, y_val, "CatBoost (24h) - tuned", n_trials=15)

print("\nEnsemble (Random Forest tuned + CatBoost tuned):")
pred_ensemble = run_ensemble(pred_rf_tuned, pred_cb_tuned, y_test, "Ensemble (24h)")

print("\nSHAP Analysis (CatBoost tuned):")
cb_model = CatBoostRegressor(**best_params_cb, verbose=False, random_seed=42)
cb_model.fit(X_train, y_train)
importance_df, _ = explain_with_shap(cb_model, X_test[:500], feature_cols)
print("Top 10 features:")
print(importance_df.head(10))

In [ ]:
=== ЗАДАЧА 2: ПРОГНОЗ МАКСИМУМА ЗА 24 ЧАСА (target_avg_cpu_24h) ===

Train size: 797581, Test size: 199695

Linear Regression (базовая модель):

==================================================
Linear Regression (24h)
==================================================
MAE:  0.106885
RMSE: 0.174208
R2:   0.646884
SVR (недооценка): 42.15% (84179/199695)
OVR (переоценка): 57.85% (115516/199695)

Random Forest (базовые параметры):

==================================================
Random Forest (24h) - default
==================================================
MAE:  0.100584
RMSE: 0.166777
R2:   0.676370
SVR (недооценка): 42.45% (84780/199695)
OVR (переоценка): 57.55% (114915/199695)

CatBoost (базовые параметры):
[I 2026-05-28 22:19:16,693] A new study created in memory with name: no-name-6e04ad32-46c8-483e-bb14-be67c28fdda8

==================================================
CatBoost (24h) - default
==================================================
MAE:  0.103024
RMSE: 0.168966
R2:   0.667819
SVR (недооценка): 42.93% (85728/199695)
OVR (переоценка): 57.07% (113967/199695)

=== Optuna тюнинг для Random Forest ===
Best trial: 0. Best value: 0.102322:   7%|▋         | 1/15 [01:05<15:12, 65.20s/it]
[I 2026-05-28 22:20:21,895] Trial 0 finished with value: 0.10232161411910845 and parameters: {'n_estimators': 250, 'max_depth': 20, 'min_samples_split': 15, 'min_samples_leaf': 6, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.10232161411910845.
Best trial: 0. Best value: 0.102322:  13%|█▎        | 2/15 [02:40<18:00, 83.10s/it]
[I 2026-05-28 22:21:57,528] Trial 1 finished with value: 0.10379956429852043 and parameters: {'n_estimators': 450, 'max_depth': 14, 'min_samples_split': 15, 'min_samples_leaf': 1, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.10232161411910845.
Best trial: 0. Best value: 0.102322:  20%|██        | 3/15 [04:12<17:25, 87.13s/it]
[I 2026-05-28 22:23:29,458] Trial 2 finished with value: 0.10797994683560697 and parameters: {'n_estimators': 150, 'max_depth': 7, 'min_samples_split': 7, 'min_samples_leaf': 6, 'max_features': None}. Best is trial 0 with value: 0.10232161411910845.
Best trial: 0. Best value: 0.102322:  27%|██▋       | 4/15 [04:37<11:27, 62.48s/it]
[I 2026-05-28 22:23:54,142] Trial 3 finished with value: 0.10682814320215639 and parameters: {'n_estimators': 150, 'max_depth': 9, 'min_samples_split': 8, 'min_samples_leaf': 5, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.10232161411910845.
Best trial: 0. Best value: 0.102322:  33%|███▎      | 5/15 [07:25<16:45, 100.54s/it]
[I 2026-05-28 22:26:42,180] Trial 4 finished with value: 0.11072474115402592 and parameters: {'n_estimators': 350, 'max_depth': 5, 'min_samples_split': 13, 'min_samples_leaf': 2, 'max_features': None}. Best is trial 0 with value: 0.10232161411910845.
Best trial: 0. Best value: 0.102322:  40%|████      | 6/15 [12:54<26:43, 178.13s/it]
[I 2026-05-28 22:32:10,917] Trial 5 finished with value: 0.10606470993727561 and parameters: {'n_estimators': 450, 'max_depth': 9, 'min_samples_split': 3, 'min_samples_leaf': 7, 'max_features': None}. Best is trial 0 with value: 0.10232161411910845.
Best trial: 0. Best value: 0.102322:  47%|████▋     | 7/15 [14:54<21:14, 159.32s/it]
[I 2026-05-28 22:34:11,522] Trial 6 finished with value: 0.1023219713493849 and parameters: {'n_estimators': 100, 'max_depth': 19, 'min_samples_split': 6, 'min_samples_leaf': 7, 'max_features': None}. Best is trial 0 with value: 0.10232161411910845.
Best trial: 7. Best value: 0.102314:  53%|█████▎    | 8/15 [17:55<19:22, 166.08s/it]
[I 2026-05-28 22:37:12,061] Trial 7 finished with value: 0.10231423799074696 and parameters: {'n_estimators': 150, 'max_depth': 20, 'min_samples_split': 16, 'min_samples_leaf': 10, 'max_features': None}. Best is trial 7 with value: 0.10231423799074696.
Best trial: 7. Best value: 0.102314:  60%|██████    | 9/15 [19:05<13:35, 135.98s/it]
[I 2026-05-28 22:38:21,864] Trial 8 finished with value: 0.1069294738334483 and parameters: {'n_estimators': 100, 'max_depth': 8, 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_features': None}. Best is trial 7 with value: 0.10231423799074696.
Best trial: 7. Best value: 0.102314:  67%|██████▋   | 10/15 [22:09<12:35, 151.01s/it]
[I 2026-05-28 22:41:26,525] Trial 9 finished with value: 0.1061168684152943 and parameters: {'n_estimators': 250, 'max_depth': 9, 'min_samples_split': 12, 'min_samples_leaf': 2, 'max_features': None}. Best is trial 7 with value: 0.10231423799074696.
Best trial: 7. Best value: 0.102314:  73%|███████▎  | 11/15 [23:30<08:37, 129.44s/it]
[I 2026-05-28 22:42:47,045] Trial 10 finished with value: 0.10315707852395221 and parameters: {'n_estimators': 350, 'max_depth': 16, 'min_samples_split': 20, 'min_samples_leaf': 10, 'max_features': 'log2'}. Best is trial 7 with value: 0.10231423799074696.
Best trial: 7. Best value: 0.102314:  80%|████████  | 12/15 [24:34<05:28, 109.53s/it]
[I 2026-05-28 22:43:51,041] Trial 11 finished with value: 0.10257545621336982 and parameters: {'n_estimators': 250, 'max_depth': 20, 'min_samples_split': 17, 'min_samples_leaf': 10, 'max_features': 'sqrt'}. Best is trial 7 with value: 0.10231423799074696.
Best trial: 7. Best value: 0.102314:  87%|████████▋ | 13/15 [25:22<03:02, 91.07s/it]
[I 2026-05-28 22:44:39,640] Trial 12 finished with value: 0.10290404619867932 and parameters: {'n_estimators': 200, 'max_depth': 17, 'min_samples_split': 18, 'min_samples_leaf': 8, 'max_features': 'log2'}. Best is trial 7 with value: 0.10231423799074696.
Best trial: 7. Best value: 0.102314:  93%|█████████▎| 14/15 [26:36<01:25, 85.90s/it]
[I 2026-05-28 22:45:53,601] Trial 13 finished with value: 0.10273128067742 and parameters: {'n_estimators': 300, 'max_depth': 18, 'min_samples_split': 15, 'min_samples_leaf': 9, 'max_features': 'sqrt'}. Best is trial 7 with value: 0.10231423799074696.
Best trial: 7. Best value: 0.102314: 100%|██████████| 15/15 [27:18<00:00, 109.23s/it]
[I 2026-05-28 22:46:35,145] Trial 14 finished with value: 0.10413485339195896 and parameters: {'n_estimators': 200, 'max_depth': 13, 'min_samples_split': 10, 'min_samples_leaf': 4, 'max_features': 'sqrt'}. Best is trial 7 with value: 0.10231423799074696.
Best params: {'n_estimators': 150, 'max_depth': 20, 'min_samples_split': 16, 'min_samples_leaf': 10, 'max_features': None}
Best val MAE: 0.102314
[I 2026-05-28 22:50:13,337] A new study created in memory with name: no-name-26182d62-f920-43d7-a761-7f96b27d8355

==================================================
Random Forest (24h) - tuned
==================================================
MAE:  0.099904
RMSE: 0.166317
R2:   0.678150
SVR (недооценка): 42.57% (85005/199695)
OVR (переоценка): 57.41% (114643/199695)

=== Optuna тюнинг для CatBoost ===
Best trial: 0. Best value: 0.103886:   7%|▋         | 1/15 [00:16<03:50, 16.43s/it]
[I 2026-05-28 22:50:29,769] Trial 0 finished with value: 0.10388626451885194 and parameters: {'iterations': 350, 'depth': 10, 'learning_rate': 0.05395030966670229, 'l2_leaf_reg': 6.387926357773329}. Best is trial 0 with value: 0.10388626451885194.
Best trial: 0. Best value: 0.103886:  13%|█▎        | 2/15 [00:21<02:09,  9.98s/it]
[I 2026-05-28 22:50:35,240] Trial 1 finished with value: 0.11440272716400451 and parameters: {'iterations': 250, 'depth': 5, 'learning_rate': 0.011430983876313222, 'l2_leaf_reg': 8.795585311974417}. Best is trial 0 with value: 0.10388626451885194.
Best trial: 0. Best value: 0.103886:  20%|██        | 3/15 [00:34<02:16, 11.34s/it]
[I 2026-05-28 22:50:48,202] Trial 2 finished with value: 0.1078650546252039 and parameters: {'iterations': 450, 'depth': 8, 'learning_rate': 0.010485387725194618, 'l2_leaf_reg': 9.72918866945795}. Best is trial 0 with value: 0.10388626451885194.
Best trial: 0. Best value: 0.103886:  27%|██▋       | 4/15 [00:45<02:03, 11.19s/it]
[I 2026-05-28 22:50:59,163] Trial 3 finished with value: 0.1077461567711378 and parameters: {'iterations': 550, 'depth': 5, 'learning_rate': 0.015199348301309814, 'l2_leaf_reg': 2.650640588680904}. Best is trial 0 with value: 0.10388626451885194.
Best trial: 0. Best value: 0.103886:  33%|███▎      | 5/15 [00:53<01:38,  9.87s/it]
[I 2026-05-28 22:51:06,698] Trial 4 finished with value: 0.10661669985942916 and parameters: {'iterations': 300, 'depth': 7, 'learning_rate': 0.027036160666620016, 'l2_leaf_reg': 3.6210622617823773}. Best is trial 0 with value: 0.10388626451885194.
Best trial: 0. Best value: 0.103886:  40%|████      | 6/15 [01:01<01:23,  9.26s/it]
[I 2026-05-28 22:51:14,772] Trial 5 finished with value: 0.10832425703478613 and parameters: {'iterations': 450, 'depth': 4, 'learning_rate': 0.019594972058679168, 'l2_leaf_reg': 4.297256589643226}. Best is trial 0 with value: 0.10388626451885194.
Best trial: 0. Best value: 0.103886:  47%|████▋     | 7/15 [01:15<01:27, 10.93s/it]
[I 2026-05-28 22:51:29,124] Trial 6 finished with value: 0.10625755384522656 and parameters: {'iterations': 400, 'depth': 9, 'learning_rate': 0.015837031559118753, 'l2_leaf_reg': 5.628109945722504}. Best is trial 0 with value: 0.10388626451885194.
Best trial: 0. Best value: 0.103886:  53%|█████▎    | 8/15 [01:23<01:10, 10.02s/it]
[I 2026-05-28 22:51:37,197] Trial 7 finished with value: 0.10714351362538091 and parameters: {'iterations': 450, 'depth': 4, 'learning_rate': 0.04050837781329675, 'l2_leaf_reg': 2.5347171131856236}. Best is trial 0 with value: 0.10388626451885194.
Best trial: 0. Best value: 0.103886:  60%|██████    | 9/15 [01:33<00:58,  9.82s/it]
[I 2026-05-28 22:51:46,572] Trial 8 finished with value: 0.10406604226273168 and parameters: {'iterations': 200, 'depth': 10, 'learning_rate': 0.0923915031962725, 'l2_leaf_reg': 8.275576133048151}. Best is trial 0 with value: 0.10388626451885194.
Best trial: 0. Best value: 0.103886:  67%|██████▋   | 10/15 [01:38<00:42,  8.52s/it]
[I 2026-05-28 22:51:52,182] Trial 9 finished with value: 0.10753815722109385 and parameters: {'iterations': 300, 'depth': 4, 'learning_rate': 0.04833180632488466, 'l2_leaf_reg': 4.961372443656412}. Best is trial 0 with value: 0.10388626451885194.
Best trial: 10. Best value: 0.102874:  73%|███████▎  | 11/15 [02:06<00:57, 14.29s/it]
[I 2026-05-28 22:52:19,573] Trial 10 finished with value: 0.10287402832091673 and parameters: {'iterations': 600, 'depth': 10, 'learning_rate': 0.07224943768187977, 'l2_leaf_reg': 6.758183738140614}. Best is trial 10 with value: 0.10287402832091673.
Best trial: 11. Best value: 0.102859:  80%|████████  | 12/15 [02:33<00:55, 18.36s/it]
[I 2026-05-28 22:52:47,219] Trial 11 finished with value: 0.10285914053805058 and parameters: {'iterations': 600, 'depth': 10, 'learning_rate': 0.07515866857795253, 'l2_leaf_reg': 6.741745796899547}. Best is trial 11 with value: 0.10285914053805058.
Best trial: 11. Best value: 0.102859:  87%|████████▋ | 13/15 [02:50<00:35, 17.77s/it]
[I 2026-05-28 22:53:03,637] Trial 12 finished with value: 0.10331675020245977 and parameters: {'iterations': 600, 'depth': 8, 'learning_rate': 0.09858632185164357, 'l2_leaf_reg': 7.0164298800690945}. Best is trial 11 with value: 0.10285914053805058.
Best trial: 11. Best value: 0.102859:  93%|█████████▎| 14/15 [03:10<00:18, 18.59s/it]
[I 2026-05-28 22:53:24,120] Trial 13 finished with value: 0.10337698906319885 and parameters: {'iterations': 600, 'depth': 9, 'learning_rate': 0.06507547631899815, 'l2_leaf_reg': 7.227974223127852}. Best is trial 11 with value: 0.10285914053805058.
Best trial: 11. Best value: 0.102859: 100%|██████████| 15/15 [03:35<00:00, 14.38s/it]
[I 2026-05-28 22:53:49,030] Trial 14 finished with value: 0.10306377386990744 and parameters: {'iterations': 550, 'depth': 10, 'learning_rate': 0.07105089399830998, 'l2_leaf_reg': 7.836931920718491}. Best is trial 11 with value: 0.10285914053805058.
Best params: {'iterations': 600, 'depth': 10, 'learning_rate': 0.07515866857795253, 'l2_leaf_reg': 6.741745796899547}
Best val MAE: 0.102859

==================================================
CatBoost (24h) - tuned
==================================================
MAE:  0.099975
RMSE: 0.165810
R2:   0.680109
SVR (недооценка): 43.33% (86536/199695)
OVR (переоценка): 56.67% (113159/199695)

Ensemble (Random Forest tuned + CatBoost tuned):

==================================================
Ensemble (24h)
==================================================
MAE:  0.099649
RMSE: 0.165517
R2:   0.681238
SVR (недооценка): 42.78% (85420/199695)
OVR (переоценка): 57.22% (114275/199695)

SHAP Analysis (CatBoost tuned):

Top 10 features:
                        feature  shap_importance
2             avg_cpu_mean_0_1d         0.037900
11  avg_cpu_history_mean_before         0.033378
6                avg_cpu_lag_5m         0.026896
3             avg_cpu_mean_1_2d         0.024671
5             avg_cpu_mean_3_4d         0.019975
9               avg_cpu_lag_20m         0.018132
7               avg_cpu_lag_10m         0.014541
8               avg_cpu_lag_15m         0.013426
4             avg_cpu_mean_2_3d         0.011309
0                   maximum_cpu         0.006300

In [ ]:
train_df = pd.read_parquet('FE_google/features_result/train_df_google.parquet')
test_df = pd.read_parquet('FE_google/features_result/test_df_google.parquet')

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")

feature_cols_lifetime = [
    'maximum_cpu', 'maximum_memory',
    'avg_cpu_lag_5m', 'avg_cpu_lag_10m', 'avg_cpu_lag_15m', 'avg_cpu_lag_20m',
    'avg_cpu_autocorr_lag1_before',
    'requested_cpu', 'requested_ram'
]

print(f"Используется признаков: {len(feature_cols_lifetime)}")
print(f"Признаки: {feature_cols_lifetime}")

In [ ]:
print("=== ЗАДАЧА 3: ПРОГНОЗ НА ВЕСЬ СРОК ЖИЗНИ (target_avg_cpu_all) ===\n")

target_col = 'target_avg_cpu_all'

X_train, X_test, y_train, y_test, imputer, scaler = prepare_features(train_df, test_df, feature_cols_lifetime, target_col)

print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")

X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

print("\nRidge Regression (базовая модель):")
pred_ridge = run_ridge_regression(X_train, X_test, y_train, y_test, alpha=1.0, name="Ridge (lifetime)")

print("\nRandom Forest (базовые параметры):")
pred_rf_default = run_random_forest_default(X_train, X_test, y_train, y_test, "Random Forest (lifetime) - default")

print("\nCatBoost (базовые параметры):")
pred_cb_default = run_catboost_default(X_train, X_test, y_train, y_test, "CatBoost (lifetime) - default")

print("\n=== Optuna тюнинг для Random Forest ===")
pred_rf_tuned, best_params_rf = run_tuned_random_forest(X_train, X_test, y_train, y_test, X_tr, y_tr, X_val, y_val, "Random Forest (lifetime) - tuned", n_trials=15)

print("\n=== Optuna тюнинг для CatBoost ===")
pred_cb_tuned, best_params_cb = run_tuned_catboost(X_train, X_test, y_train, y_test, X_tr, y_tr, X_val, y_val, "CatBoost (lifetime) - tuned", n_trials=15)

print("\nEnsemble (Random Forest tuned + CatBoost tuned + Ridge):")
pred_ensemble = run_ensemble_three(pred_rf_tuned, pred_cb_tuned, pred_ridge, y_test, "Ensemble (lifetime)")

print("\nSHAP Analysis (CatBoost tuned):")
cb_model = CatBoostRegressor(**best_params_cb, verbose=False, random_seed=42)
cb_model.fit(X_train, y_train)
importance_df, _ = explain_with_shap(cb_model, X_test[:500], feature_cols_lifetime)
print("Top 10 features:")
print(importance_df.head(10))

In [ ]:
=== ЗАДАЧА 3: ПРОГНОЗ НА ВЕСЬ СРОК ЖИЗНИ (target_avg_cpu_all) ===

Train size: 797581, Test size: 199695

Ridge Regression (базовая модель):

==================================================
Ridge (lifetime)
==================================================
MAE:  0.085924
RMSE: 0.118276
R2:   0.747851
SVR (недооценка): 41.15% (82168/199695)
OVR (переоценка): 58.85% (117527/199695)

Random Forest (базовые параметры):

==================================================
Random Forest (lifetime) - default
==================================================
MAE:  0.033754
RMSE: 0.060765
R2:   0.933446
SVR (недооценка): 45.98% (91817/199695)
OVR (переоценка): 54.02% (107878/199695)

CatBoost (базовые параметры):
[I 2026-05-29 00:45:31,868] A new study created in memory with name: no-name-d682dbdf-20db-4943-a6b2-a149f8fe7c8d

==================================================
CatBoost (lifetime) - default
==================================================
MAE:  0.053762
RMSE: 0.078987
R2:   0.887544
SVR (недооценка): 46.57% (93005/199695)
OVR (переоценка): 53.43% (106690/199695)

=== Optuna тюнинг для Random Forest ===
Best trial: 0. Best value: 0.0275941:   7%|▋         | 1/15 [00:55<12:51, 55.10s/it]
[I 2026-05-29 00:46:26,967] Trial 0 finished with value: 0.027594086262315666 and parameters: {'n_estimators': 250, 'max_depth': 20, 'min_samples_split': 15, 'min_samples_leaf': 6, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.027594086262315666.
Best trial: 0. Best value: 0.0275941:  13%|█▎        | 2/15 [02:17<15:22, 70.97s/it]
[I 2026-05-29 00:47:49,051] Trial 1 finished with value: 0.03783692764323978 and parameters: {'n_estimators': 450, 'max_depth': 14, 'min_samples_split': 15, 'min_samples_leaf': 1, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.027594086262315666.
Best trial: 0. Best value: 0.0275941:  20%|██        | 3/15 [03:04<12:01, 60.09s/it]
[I 2026-05-29 00:48:36,196] Trial 2 finished with value: 0.06588598552420745 and parameters: {'n_estimators': 150, 'max_depth': 7, 'min_samples_split': 7, 'min_samples_leaf': 6, 'max_features': None}. Best is trial 0 with value: 0.027594086262315666.
Best trial: 0. Best value: 0.0275941:  27%|██▋       | 4/15 [03:25<08:10, 44.60s/it]
[I 2026-05-29 00:48:57,034] Trial 3 finished with value: 0.058310261580983076 and parameters: {'n_estimators': 150, 'max_depth': 9, 'min_samples_split': 8, 'min_samples_leaf': 5, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.027594086262315666.
Best trial: 0. Best value: 0.0275941:  33%|███▎      | 5/15 [04:49<09:50, 59.10s/it]
[I 2026-05-29 00:50:21,848] Trial 4 finished with value: 0.07760647156060373 and parameters: {'n_estimators': 350, 'max_depth': 5, 'min_samples_split': 13, 'min_samples_leaf': 2, 'max_features': None}. Best is trial 0 with value: 0.027594086262315666.
Best trial: 0. Best value: 0.0275941:  40%|████      | 6/15 [07:38<14:27, 96.35s/it]
[I 2026-05-29 00:53:10,519] Trial 5 finished with value: 0.05424884940122453 and parameters: {'n_estimators': 450, 'max_depth': 9, 'min_samples_split': 3, 'min_samples_leaf': 7, 'max_features': None}. Best is trial 0 with value: 0.027594086262315666.
Best trial: 6. Best value: 0.0232366:  47%|████▋     | 7/15 [08:40<11:20, 85.02s/it]
[I 2026-05-29 00:54:12,204] Trial 6 finished with value: 0.023236625455180356 and parameters: {'n_estimators': 100, 'max_depth': 19, 'min_samples_split': 6, 'min_samples_leaf': 7, 'max_features': None}. Best is trial 6 with value: 0.023236625455180356.
Best trial: 7. Best value: 0.0230999:  53%|█████▎    | 8/15 [10:11<10:09, 87.10s/it]
[I 2026-05-29 00:55:43,758] Trial 7 finished with value: 0.023099906281281384 and parameters: {'n_estimators': 150, 'max_depth': 20, 'min_samples_split': 16, 'min_samples_leaf': 10, 'max_features': None}. Best is trial 7 with value: 0.023099906281281384.
Best trial: 7. Best value: 0.0230999:  60%|██████    | 9/15 [10:47<07:05, 70.89s/it]
[I 2026-05-29 00:56:19,019] Trial 8 finished with value: 0.06004549789271827 and parameters: {'n_estimators': 100, 'max_depth': 8, 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_features': None}. Best is trial 7 with value: 0.023099906281281384.
Best trial: 7. Best value: 0.0230999:  67%|██████▋   | 10/15 [12:22<06:31, 78.31s/it]
[I 2026-05-29 00:57:53,921] Trial 9 finished with value: 0.05429488378735279 and parameters: {'n_estimators': 250, 'max_depth': 9, 'min_samples_split': 12, 'min_samples_leaf': 2, 'max_features': None}. Best is trial 7 with value: 0.023099906281281384.
Best trial: 7. Best value: 0.0230999:  73%|███████▎  | 11/15 [13:30<05:01, 75.34s/it]
[I 2026-05-29 00:59:02,521] Trial 10 finished with value: 0.03389812953000759 and parameters: {'n_estimators': 350, 'max_depth': 16, 'min_samples_split': 20, 'min_samples_leaf': 10, 'max_features': 'log2'}. Best is trial 7 with value: 0.023099906281281384.
Best trial: 7. Best value: 0.0230999:  80%|████████  | 12/15 [13:52<02:57, 59.20s/it]
[I 2026-05-29 00:59:24,824] Trial 11 finished with value: 0.028974819204266946 and parameters: {'n_estimators': 100, 'max_depth': 20, 'min_samples_split': 20, 'min_samples_leaf': 10, 'max_features': 'log2'}. Best is trial 7 with value: 0.023099906281281384.
Best trial: 7. Best value: 0.0230999:  87%|████████▋ | 13/15 [15:45<02:30, 75.36s/it]
[I 2026-05-29 01:01:17,353] Trial 12 finished with value: 0.026164074505520055 and parameters: {'n_estimators': 200, 'max_depth': 17, 'min_samples_split': 7, 'min_samples_leaf': 8, 'max_features': None}. Best is trial 7 with value: 0.023099906281281384.
Best trial: 7. Best value: 0.0230999:  93%|█████████▎| 14/15 [16:45<01:10, 70.60s/it]
[I 2026-05-29 01:02:16,969] Trial 13 finished with value: 0.024757197095506413 and parameters: {'n_estimators': 100, 'max_depth': 18, 'min_samples_split': 17, 'min_samples_leaf': 8, 'max_features': None}. Best is trial 7 with value: 0.023099906281281384.
Best trial: 7. Best value: 0.0230999: 100%|██████████| 15/15 [18:21<00:00, 73.42s/it]
[I 2026-05-29 01:03:53,139] Trial 14 finished with value: 0.036377472875496836 and parameters: {'n_estimators': 200, 'max_depth': 13, 'min_samples_split': 9, 'min_samples_leaf': 9, 'max_features': None}. Best is trial 7 with value: 0.023099906281281384.
Best params: {'n_estimators': 150, 'max_depth': 20, 'min_samples_split': 16, 'min_samples_leaf': 10, 'max_features': None}
Best val MAE: 0.023100
[I 2026-05-29 01:05:41,650] A new study created in memory with name: no-name-01de763a-0960-4b3a-bb18-92f527b5e0ba

==================================================
Random Forest (lifetime) - tuned
==================================================
MAE:  0.028451
RMSE: 0.055190
R2:   0.945098
SVR (недооценка): 46.09% (92035/199695)
OVR (переоценка): 53.91% (107660/199695)

=== Optuna тюнинг для CatBoost ===
Best trial: 0. Best value: 0.0369743:   7%|▋         | 1/15 [00:14<03:17, 14.12s/it]
[I 2026-05-29 01:05:55,767] Trial 0 finished with value: 0.036974298075248574 and parameters: {'iterations': 350, 'depth': 10, 'learning_rate': 0.05395030966670229, 'l2_leaf_reg': 6.387926357773329}. Best is trial 0 with value: 0.036974298075248574.
Best trial: 0. Best value: 0.0369743:  13%|█▎        | 2/15 [00:18<01:52,  8.64s/it]
[I 2026-05-29 01:06:00,572] Trial 1 finished with value: 0.07970196849906962 and parameters: {'iterations': 250, 'depth': 5, 'learning_rate': 0.011430983876313222, 'l2_leaf_reg': 8.795585311974417}. Best is trial 0 with value: 0.036974298075248574.
Best trial: 0. Best value: 0.0369743:  20%|██        | 3/15 [00:29<01:56,  9.71s/it]
[I 2026-05-29 01:06:11,551] Trial 2 finished with value: 0.06081957304212535 and parameters: {'iterations': 450, 'depth': 8, 'learning_rate': 0.010485387725194618, 'l2_leaf_reg': 9.72918866945795}. Best is trial 0 with value: 0.036974298075248574.
Best trial: 0. Best value: 0.0369743:  27%|██▋       | 4/15 [00:39<01:47,  9.76s/it]
[I 2026-05-29 01:06:21,365] Trial 3 finished with value: 0.06501664004196263 and parameters: {'iterations': 550, 'depth': 5, 'learning_rate': 0.015199348301309814, 'l2_leaf_reg': 2.650640588680904}. Best is trial 0 with value: 0.036974298075248574.
Best trial: 0. Best value: 0.0369743:  33%|███▎      | 5/15 [00:46<01:26,  8.63s/it]
[I 2026-05-29 01:06:28,012] Trial 4 finished with value: 0.05737970305665848 and parameters: {'iterations': 300, 'depth': 7, 'learning_rate': 0.027036160666620016, 'l2_leaf_reg': 3.6210622617823773}. Best is trial 0 with value: 0.036974298075248574.
Best trial: 0. Best value: 0.0369743:  40%|████      | 6/15 [00:53<01:14,  8.24s/it]
[I 2026-05-29 01:06:35,502] Trial 5 finished with value: 0.06867092178469435 and parameters: {'iterations': 450, 'depth': 4, 'learning_rate': 0.019594972058679168, 'l2_leaf_reg': 4.297256589643226}. Best is trial 0 with value: 0.036974298075248574.
Best trial: 0. Best value: 0.0369743:  47%|████▋     | 7/15 [01:05<01:15,  9.44s/it]
[I 2026-05-29 01:06:47,420] Trial 6 finished with value: 0.05340486382617726 and parameters: {'iterations': 400, 'depth': 9, 'learning_rate': 0.015837031559118753, 'l2_leaf_reg': 5.628109945722504}. Best is trial 0 with value: 0.036974298075248574.
Best trial: 0. Best value: 0.0369743:  53%|█████▎    | 8/15 [01:13<01:01,  8.78s/it]
[I 2026-05-29 01:06:54,791] Trial 7 finished with value: 0.061434096964765615 and parameters: {'iterations': 450, 'depth': 4, 'learning_rate': 0.04050837781329675, 'l2_leaf_reg': 2.5347171131856236}. Best is trial 0 with value: 0.036974298075248574.
Best trial: 0. Best value: 0.0369743:  60%|██████    | 9/15 [01:21<00:51,  8.58s/it]
[I 2026-05-29 01:07:02,936] Trial 8 finished with value: 0.037289129222427025 and parameters: {'iterations': 200, 'depth': 10, 'learning_rate': 0.0923915031962725, 'l2_leaf_reg': 8.275576133048151}. Best is trial 0 with value: 0.036974298075248574.
Best trial: 0. Best value: 0.0369743:  67%|██████▋   | 10/15 [01:26<00:37,  7.52s/it]
[I 2026-05-29 01:07:08,087] Trial 9 finished with value: 0.06368414286690728 and parameters: {'iterations': 300, 'depth': 4, 'learning_rate': 0.04833180632488466, 'l2_leaf_reg': 4.961372443656412}. Best is trial 0 with value: 0.036974298075248574.
Best trial: 10. Best value: 0.0288243:  73%|███████▎  | 11/15 [01:50<00:50, 12.55s/it]
[I 2026-05-29 01:07:32,045] Trial 10 finished with value: 0.028824254460295837 and parameters: {'iterations': 600, 'depth': 10, 'learning_rate': 0.07224943768187977, 'l2_leaf_reg': 6.758183738140614}. Best is trial 10 with value: 0.028824254460295837.
Best trial: 11. Best value: 0.028482:  80%|████████  | 12/15 [02:14<00:48, 16.08s/it]
[I 2026-05-29 01:07:56,178] Trial 11 finished with value: 0.028482049940782465 and parameters: {'iterations': 600, 'depth': 10, 'learning_rate': 0.07515866857795253, 'l2_leaf_reg': 6.741745796899547}. Best is trial 11 with value: 0.028482049940782465.
Best trial: 11. Best value: 0.028482:  87%|████████▋ | 13/15 [02:29<00:31, 15.61s/it]
[I 2026-05-29 01:08:10,700] Trial 12 finished with value: 0.03126397074022714 and parameters: {'iterations': 600, 'depth': 8, 'learning_rate': 0.09858632185164357, 'l2_leaf_reg': 7.0164298800690945}. Best is trial 11 with value: 0.028482049940782465.
Best trial: 11. Best value: 0.028482:  93%|█████████▎| 14/15 [02:46<00:16, 16.07s/it]
[I 2026-05-29 01:08:27,837] Trial 13 finished with value: 0.03220155743724239 and parameters: {'iterations': 600, 'depth': 9, 'learning_rate': 0.06507547631899815, 'l2_leaf_reg': 7.227974223127852}. Best is trial 11 with value: 0.028482049940782465.
Best trial: 11. Best value: 0.028482: 100%|██████████| 15/15 [03:07<00:00, 12.53s/it]
[I 2026-05-29 01:08:49,628] Trial 14 finished with value: 0.029738424751825154 and parameters: {'iterations': 550, 'depth': 10, 'learning_rate': 0.07105089399830998, 'l2_leaf_reg': 7.836931920718491}. Best is trial 11 with value: 0.028482049940782465.
Best params: {'iterations': 600, 'depth': 10, 'learning_rate': 0.07515866857795253, 'l2_leaf_reg': 6.741745796899547}
Best val MAE: 0.028482

==================================================
CatBoost (lifetime) - tuned
==================================================
MAE:  0.032181
RMSE: 0.053949
R2:   0.947539
SVR (недооценка): 49.48% (98813/199695)
OVR (переоценка): 50.52% (100882/199695)

Ensemble (Random Forest tuned + CatBoost tuned + Ridge):

==================================================
Ensemble (lifetime)
==================================================
MAE:  0.044458
RMSE: 0.065087
R2:   0.923642
SVR (недооценка): 42.06% (83983/199695)
OVR (переоценка): 57.94% (115712/199695)

SHAP Analysis (CatBoost tuned):

Top 10 features:
                        feature  shap_importance
0                   maximum_cpu         0.035904
5               avg_cpu_lag_20m         0.033223
2                avg_cpu_lag_5m         0.032132
4               avg_cpu_lag_15m         0.026869
8                 requested_ram         0.026633
7                 requested_cpu         0.024878
3               avg_cpu_lag_10m         0.022642
6  avg_cpu_autocorr_lag1_before         0.016116
1                maximum_memory         0.004302